In [1]:
from sklearn.datasets import fetch_california_housing
import numpy as np

data = fetch_california_housing(as_frame=False)
data.data

array([[   8.3252    ,   41.        ,    6.98412698, ...,    2.55555556,
          37.88      , -122.23      ],
       [   8.3014    ,   21.        ,    6.23813708, ...,    2.10984183,
          37.86      , -122.22      ],
       [   7.2574    ,   52.        ,    8.28813559, ...,    2.80225989,
          37.85      , -122.24      ],
       ...,
       [   1.7       ,   17.        ,    5.20554273, ...,    2.3256351 ,
          39.43      , -121.22      ],
       [   1.8672    ,   18.        ,    5.32951289, ...,    2.12320917,
          39.43      , -121.32      ],
       [   2.3886    ,   16.        ,    5.25471698, ...,    2.61698113,
          39.37      , -121.24      ]], shape=(20640, 8))

In [2]:
data.target

array([4.526, 3.585, 3.521, ..., 0.923, 0.847, 0.894], shape=(20640,))

In [3]:
X_train, y_train = data.data[:15000], data.target[:15000]
X_valid, y_valid = X_train[:5000], y_train[:5000]
X_test, y_test = data.data[15000:], data.target[15000:]

Transforming and scaling data (This can also be done using StandardScaler which implements the same formula)

In [4]:
import torch
X_train = torch.FloatTensor(X_train)
X_valid= torch.FloatTensor(X_valid)
X_test = torch.FloatTensor(X_test)
means = X_train.mean(dim=0,keepdim=True)
std = X_train.std(dim=0,keepdim=True)

In [5]:
print(f"means:{means}") # mean of each column 
print(f"std:{std}") # std of each column

means:tensor([[ 3.7969e+00,  2.9077e+01,  5.4070e+00,  1.1047e+00,  1.4380e+03,
          3.0305e+00,  3.5190e+01, -1.1904e+02]])
std:tensor([[1.8915e+00, 1.2368e+01, 2.7607e+00, 5.3685e-01, 1.1389e+03, 5.3644e+00,
         2.0452e+00, 1.8377e+00]])


In [6]:
X_train_scaled = (X_train - means) / (std + 1e-8)
X_valid_scaled = (X_valid - means) / (std + 1e-8)
X_test_scaled = (X_test - means) / (std + 1e-8)

Since our predictions will be column vectors (i.e., matrices with a single column), we need to ensure that our targets are also column vectors

In [7]:
y_train = torch.FloatTensor(y_train).reshape(-1, 1)
y_valid = torch.FloatTensor(y_valid).reshape(-1, 1)
y_test = torch.FloatTensor(y_test).reshape(-1, 1)

In [8]:
torch.manual_seed(42)
n_features = X_train_scaled.shape[1]
w = torch.randn((n_features,1),requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)

X_train_scaled.shape[1]: Number of feature variables or columns ($D$) <br>
w = torch.randn((n_features,1)) specifies to generate a random number between 8 to 1 <br> requires_grad=True so Pytorch would know to track the computation graph of $w$ and $b$

We will use autodiff to compute the gradients and batch gradient descent (BGD), using the full training set at each training step

In [9]:
learning_rate = 0.4
n_epochs = 20
for epoch in range(n_epochs):
    y_pred = X_train_scaled @ w + b # Equation of linear regression
    loss_mse = ((y_pred - y_train)**2).mean()
    loss_mse.backward()
    with torch.no_grad():
        b -= learning_rate * b.grad
        w -= learning_rate * w.grad
        b.grad.zero_()
        w.grad.zero_()
    print(f"Epoch: {epoch+1}/{n_epochs}, Loss: {loss_mse.item()}")

Epoch: 1/20, Loss: 16.213895797729492
Epoch: 2/20, Loss: 5.12765645980835
Epoch: 3/20, Loss: 2.4164185523986816
Epoch: 4/20, Loss: 1.4086052179336548
Epoch: 5/20, Loss: 0.988771915435791
Epoch: 6/20, Loss: 0.8021609783172607
Epoch: 7/20, Loss: 0.7134009599685669
Epoch: 8/20, Loss: 0.6672422885894775
Epoch: 9/20, Loss: 0.6402928233146667
Epoch: 10/20, Loss: 0.6224126219749451
Epoch: 11/20, Loss: 0.609121561050415
Epoch: 12/20, Loss: 0.5983965396881104
Epoch: 13/20, Loss: 0.5892924666404724
Epoch: 14/20, Loss: 0.5813428163528442
Epoch: 15/20, Loss: 0.5742966532707214
Epoch: 16/20, Loss: 0.5680025219917297
Epoch: 17/20, Loss: 0.5623570680618286
Epoch: 18/20, Loss: 0.5572821497917175
Epoch: 19/20, Loss: 0.552713930606842
Epoch: 20/20, Loss: 0.5485981106758118


In [10]:
X_new = X_test_scaled[:3]
with torch.no_grad():
    y_pred = X_new @ w + b
y_pred

tensor([[2.1124],
        [1.1764],
        [0.9760]])

In [11]:
y_test[:3]

tensor([[1.4070],
        [1.1920],
        [1.3200]])

In [12]:
from torch import nn
torch.manual_seed(42)
model = nn.Linear(in_features=n_features,out_features=1)
print(model.bias)
print(model.weight)

Parameter containing:
tensor([0.3117], requires_grad=True)
Parameter containing:
tensor([[ 0.2703,  0.2935, -0.0828,  0.3248, -0.0775,  0.0713, -0.1721,  0.2076]],
       requires_grad=True)


Notice that both parameters were automatically initialized randomly (which is why we used manual_seed() to get reproducible results)

In [13]:
for params in model.parameters():
    print(params)

Parameter containing:
tensor([[ 0.2703,  0.2935, -0.0828,  0.3248, -0.0775,  0.0713, -0.1721,  0.2076]],
       requires_grad=True)
Parameter containing:
tensor([0.3117], requires_grad=True)


Parameter() method returns an iterator over all the model's attributes of type Parameter and same for its Submodules and doesn't return a regular tensor even for those with require_grad = True

In [14]:
for name,params in model.named_parameters():
    print(f"Name:{name} , Parameters:{params}")

Name:weight , Parameters:Parameter containing:
tensor([[ 0.2703,  0.2935, -0.0828,  0.3248, -0.0775,  0.0713, -0.1721,  0.2076]],
       requires_grad=True)
Name:bias , Parameters:Parameter containing:
tensor([0.3117], requires_grad=True)


PyTorch provides a few different optimizers (we will discuss them in the next chapter). Here we’re using the simple stochastic gradient descent (SGD) optimizer, which can be used for SGD, mini-batch GD, or batch gradient descent. To initialize it, we must give it the model parameters and the learning rate.

For the loss function, we create an instance of the nn.MSELoss class: this is also a module, so we can use it like a function, giving it the predictions and the targets, and it will compute the MSE. The nn module contains many other loss functions and other neural net tools

In [15]:
optimizer = torch.optim.SGD(model.parameters(),lr=learning_rate) #i.e. = 0.4
mse = nn.MSELoss()

In [16]:
def train_bgd(model, optimizer, criterion, X_train, y_train, n_epochs):
    for epoch in range(n_epochs):
        y_pred = model(X_train)
        loss = criterion(y_pred, y_train)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        print(f"Epoch: {epoch+1}/{n_epochs}, Loss: {loss.item()}")

In [17]:
train_bgd(model, optimizer, mse, X_train_scaled, y_train, 20)

Epoch: 1/20, Loss: 3.8578336238861084
Epoch: 2/20, Loss: 0.7024627327919006
Epoch: 3/20, Loss: 0.5736716389656067
Epoch: 4/20, Loss: 0.5620139241218567
Epoch: 5/20, Loss: 0.5561234354972839
Epoch: 6/20, Loss: 0.5511719584465027
Epoch: 7/20, Loss: 0.5467990636825562
Epoch: 8/20, Loss: 0.5429030060768127
Epoch: 9/20, Loss: 0.5394209027290344
Epoch: 10/20, Loss: 0.536304235458374
Epoch: 11/20, Loss: 0.5335124731063843
Epoch: 12/20, Loss: 0.5310103893280029
Epoch: 13/20, Loss: 0.5287669897079468
Epoch: 14/20, Loss: 0.5267548561096191
Epoch: 15/20, Loss: 0.5249493718147278
Epoch: 16/20, Loss: 0.5233289003372192
Epoch: 17/20, Loss: 0.5218738913536072
Epoch: 18/20, Loss: 0.520566999912262
Epoch: 19/20, Loss: 0.5193927884101868
Epoch: 20/20, Loss: 0.5183372497558594


In [18]:
X_new = X_test_scaled[:3]
with torch.no_grad():
    y_pred = model(X_new) # Here we are using the trained model
y_pred

tensor([[2.1272],
        [1.2779],
        [1.1105]])

### Regression MLP <br>
 The MLP we want to build is just that: a simple stack of modules—two hidden layers and one output layer. 

In [19]:
torch.manual_seed(42)
mlp_model = nn.Sequential(
    nn.Linear(n_features,50),
    nn.ReLU(),
    nn.Linear(50,40),
    nn.ReLU(),
    nn.Linear(40,1)
)

In [20]:
for name,params in mlp_model.named_parameters():
    print(f"{name}, {params}")

0.weight, Parameter containing:
tensor([[ 0.2703,  0.2935, -0.0828,  0.3248, -0.0775,  0.0713, -0.1721,  0.2076],
        [ 0.3117, -0.2594,  0.3073,  0.0662,  0.2612,  0.0479,  0.1705, -0.0499],
        [ 0.2725,  0.0523, -0.1651,  0.0901, -0.1629, -0.0415, -0.1436,  0.2345],
        [-0.2791, -0.1630, -0.0998, -0.2126,  0.0334, -0.3492,  0.3193, -0.3003],
        [ 0.2730,  0.0588, -0.1148,  0.2185,  0.0551,  0.2857,  0.0387, -0.1115],
        [ 0.0950, -0.0959,  0.1488,  0.3157,  0.2044, -0.1546,  0.2041,  0.0633],
        [ 0.1795, -0.2155, -0.3500, -0.1366, -0.2712,  0.2901,  0.1018,  0.1464],
        [ 0.1118, -0.0062,  0.2767, -0.2512,  0.0223, -0.2413,  0.1090, -0.1218],
        [ 0.1083, -0.0737,  0.2932, -0.2096, -0.2109, -0.2109,  0.3180,  0.1178],
        [ 0.3402, -0.2918, -0.3507, -0.2766, -0.2378,  0.1432,  0.1266,  0.2938],
        [-0.1826, -0.2410,  0.1876, -0.1429,  0.2146, -0.0839,  0.2022, -0.2747],
        [-0.1784,  0.1078,  0.0747, -0.0901,  0.2107,  0.2403, -0.

In [21]:
learning_rate = 0.1
optimizer_mlp = torch.optim.SGD(mlp_model.parameters(), lr=learning_rate)
mse = nn.MSELoss()
train_bgd(mlp_model,optimizer_mlp,mse,X_train_scaled,y_train,20)

Epoch: 1/20, Loss: 4.59269380569458
Epoch: 2/20, Loss: 1.9485915899276733
Epoch: 3/20, Loss: 0.9657654166221619
Epoch: 4/20, Loss: 0.7995327115058899
Epoch: 5/20, Loss: 0.715325117111206
Epoch: 6/20, Loss: 0.6687820553779602
Epoch: 7/20, Loss: 0.6410285234451294
Epoch: 8/20, Loss: 0.6232306361198425
Epoch: 9/20, Loss: 0.6109237670898438
Epoch: 10/20, Loss: 0.6017888188362122
Epoch: 11/20, Loss: 0.5945402979850769
Epoch: 12/20, Loss: 0.588412344455719
Epoch: 13/20, Loss: 0.5830336213111877
Epoch: 14/20, Loss: 0.5781357884407043
Epoch: 15/20, Loss: 0.573562741279602
Epoch: 16/20, Loss: 0.5692182183265686
Epoch: 17/20, Loss: 0.5650476813316345
Epoch: 18/20, Loss: 0.5610231161117554
Epoch: 19/20, Loss: 0.557138979434967
Epoch: 20/20, Loss: 0.5533733367919922


## Mini-Batch Gradient Descent using DataLoaders

In [22]:
from torch.utils.data import TensorDataset, DataLoader

train_data = TensorDataset(X_train_scaled, y_train)
train_loader = DataLoader(train_data,batch_size=64, shuffle=True) # SRS

In [23]:
print(train_data[:3])

(tensor([[ 2.3939,  0.9640,  0.5713, -0.1506, -0.9799, -0.0885,  1.3155, -1.7348],
        [ 2.3814, -0.6531,  0.3011, -0.2474,  0.8456, -0.1716,  1.3057, -1.7294],
        [ 1.8294,  1.8534,  1.0436, -0.0582, -0.8271, -0.0426,  1.3008, -1.7403]]), tensor([[4.5260],
        [3.5850],
        [3.5210]]))


In [24]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(42)
model = mlp_model.to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=0.2)
mse = nn.MSELoss()

In [25]:
def train(model, optimizer, criterion, train_loader, n_epochs):
    model.train()
    for epoch in range(n_epochs):
        total_loss = 0.0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

        mean_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch + 1}/{n_epochs}, Loss: {mean_loss:.4f}")

In [26]:
train(model, optimizer, mse, train_loader, 20)

Epoch 1/20, Loss: 0.6043
Epoch 2/20, Loss: 0.4042
Epoch 3/20, Loss: 0.3755
Epoch 4/20, Loss: 0.3475
Epoch 5/20, Loss: 0.3249
Epoch 6/20, Loss: 0.3162
Epoch 7/20, Loss: 0.3195
Epoch 8/20, Loss: 0.3031
Epoch 9/20, Loss: 0.2960
Epoch 10/20, Loss: 0.2967
Epoch 11/20, Loss: 0.2927
Epoch 12/20, Loss: 0.2829
Epoch 13/20, Loss: 0.2824
Epoch 14/20, Loss: 0.2836
Epoch 15/20, Loss: 0.2757
Epoch 16/20, Loss: 0.2804
Epoch 17/20, Loss: 0.2801
Epoch 18/20, Loss: 0.2717
Epoch 19/20, Loss: 0.2633
Epoch 20/20, Loss: 0.2670
